### Phase 2 - Strategy Optimization & Attack Iteration
Combines best spatial + frequency ideas using minimal-change attacks.
- Stepwise strategy A: small blur -> blur + slight frequency masking
- Stepwise strategy B: noise -> noise + brightness shift
- Tracks minimum change needed to flip prediction

##### Output files:

- strategy_iteration_results.csv
- strategy_stage_summary.csv
- min_change_flip.csv

In [7]:
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance

import torch
import torch.nn as nn
from torchvision import models, transforms

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "Phase2" else Path.cwd().resolve()
DATASET_ROOT = PROJECT_ROOT / "dataset"
PHASE2_DIR = PROJECT_ROOT / "Phase2"
OUT_DIR = PHASE2_DIR / "results_opt"
OUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = DATASET_ROOT / "metadata.csv"
CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2023, 0.1994, 0.2010]
IMG_SIZE = 32
TARGET_SPLIT = "test"
TARGET_CLASSES = ["FAKE", "REAL"]
NUM_SAMPLES_PER_CLASS = 24
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

def build_resnet18(num_classes=2):
    m = models.resnet18(weights=None)
    in_f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))
    return m

def find_checkpoint(root: Path) -> Path:
    cands = [
        root / "dataset" / "models" / "cifake_resnet18_final.pt",
        root / "Phase1" / "cifake_resnet18_final.pt",
    ]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError("No checkpoint found in Phase1/models or dataset/models")

def load_model(device):
    ckpt = torch.load(find_checkpoint(PROJECT_ROOT), map_location="cpu")
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model = build_resnet18()
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model

TFM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

def load_samples():
    meta = pd.read_csv(METADATA_PATH)
    rows = []
    for i, cname in enumerate(TARGET_CLASSES):
        part = meta[(meta["split"] == TARGET_SPLIT) & (meta["label_name"].str.upper() == cname)]
        rows.append(part.sample(n=min(NUM_SAMPLES_PER_CLASS, len(part)), random_state=SEED + i))
    df = pd.concat(rows, ignore_index=True)
    df["class_name"] = df["label_name"].str.upper()
    df["abs_path"] = df["filepath"].map(lambda p: str(DATASET_ROOT / p))
    return df

def pil_to_arr(img):
    return np.asarray(img, dtype=np.float32) / 255.0

def arr_to_pil(arr):
    return Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))

def predict(model, pil_img, device):
    x = TFM(pil_img).unsqueeze(0).to(device)
    with torch.inference_mode():
        probs = torch.softmax(model(x), dim=1)[0]
    pred = int(torch.argmax(probs).item())
    return pred, float(probs[1].item())

def high_freq_ratio(arr):
    f = np.fft.fftshift(np.fft.fft2(arr, axes=(0,1)), axes=(0,1))
    p = (np.abs(f) ** 2).mean(axis=2)
    h, w = p.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((y - cy) ** 2 + (x - cx) ** 2)
    rmax = np.sqrt(cy ** 2 + cx ** 2)
    mask = r > (0.66 * rmax)
    return float(p[mask].sum() / (p.sum() + 1e-12))

def blur_then_freq(img, blur_radius=0.25, mask_strength=0.08):
    b = img.filter(ImageFilter.GaussianBlur(blur_radius))
    arr = pil_to_arr(b)
    f = np.fft.fftshift(np.fft.fft2(arr, axes=(0,1)), axes=(0,1))
    h, w = arr.shape[:2]
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((y - cy) ** 2 + (x - cx) ** 2)
    rmax = np.sqrt(cy ** 2 + cx ** 2)
    hi = r > (0.65 * rmax)
    m = np.ones((h, w), dtype=np.float32)
    m[hi] -= mask_strength * rng.random(np.count_nonzero(hi), dtype=np.float32)
    return arr_to_pil(np.fft.ifft2(np.fft.ifftshift(f * m[:, :, None], axes=(0,1)), axes=(0,1)).real)

def noise_then_brightness(img, sigma=0.008, bright_shift=0.04):
    arr = pil_to_arr(img)
    arr = np.clip(arr + rng.normal(0, sigma, arr.shape), 0, 1)
    out = arr_to_pil(arr)
    return ImageEnhance.Brightness(out).enhance(1.0 + bright_shift)

def mean_abs_delta(a, b):
    return float(np.mean(np.abs(pil_to_arr(a) - pil_to_arr(b))))

print("Setup done. Output dir:", OUT_DIR)

Setup done. Output dir: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\Phase2\results_opt


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(device)
samples = load_samples()

rows = []
min_rows = []
alpha_grid = np.linspace(0.1, 1.0, 10)

for i, row in samples.iterrows():
    base = Image.open(row["abs_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    sample_id = f"{row['class_name']}_{i:03d}_{Path(row['abs_path']).stem.replace(' ', '_')}"

    base_pred, base_pfake = predict(model, base, device)
    base_hf = high_freq_ratio(pil_to_arr(base))
    base_contrast = float(np.std(pil_to_arr(base)))

    # Stepwise A: small blur -> blur + slight frequency mask
    a1 = base.filter(ImageFilter.GaussianBlur(0.25))
    a2 = blur_then_freq(base, blur_radius=0.25, mask_strength=0.08)

    # Stepwise B: noise -> noise + brightness shift
    b1 = arr_to_pil(np.clip(pil_to_arr(base) + rng.normal(0, 0.008, pil_to_arr(base).shape), 0, 1))
    b2 = noise_then_brightness(base, sigma=0.008, bright_shift=0.04)

    stages = {
        "original": base,
        "A_blur": a1,
        "A_blur_freq": a2,
        "B_noise": b1,
        "B_noise_brightness": b2,
    }

    for stage, img in stages.items():
        pred, pf = predict(model, img, device)
        rows.append({
            "sample_id": sample_id,
            "class_name": row["class_name"],
            "stage": stage,
            "p_fake": pf,
            "pred": pred,
            "base_pred": base_pred,
            "flip": int(pred != base_pred),
            "mad": mean_abs_delta(base, img),
            "hf_ratio": high_freq_ratio(pil_to_arr(img)),
            "contrast_std": float(np.std(pil_to_arr(img))),
            "hf_drop": base_hf - high_freq_ratio(pil_to_arr(img)),
            "contrast_drop": base_contrast - float(np.std(pil_to_arr(img))),
        })

    # Minimal-change search (smallest alpha that flips prediction)
    def search_min(strategy):
        for a in alpha_grid:
            if strategy == "blur_freq":
                img = blur_then_freq(base, blur_radius=0.25 * a, mask_strength=0.10 * a)
            else:
                img = noise_then_brightness(base, sigma=0.010 * a, bright_shift=0.06 * a)
            pred, _ = predict(model, img, device)
            if pred != base_pred:
                return a, mean_abs_delta(base, img)
        return np.nan, np.nan

    a_alpha, a_mad = search_min("blur_freq")
    b_alpha, b_mad = search_min("noise_brightness")

    choice = "none"
    if not np.isnan(a_mad) and (np.isnan(b_mad) or a_mad <= b_mad):
        choice = "blur_freq"
    elif not np.isnan(b_mad):
        choice = "noise_brightness"

    min_rows.append({
        "sample_id": sample_id,
        "class_name": row["class_name"],
        "base_pred": base_pred,
        "min_alpha_blur_freq": a_alpha,
        "min_mad_blur_freq": a_mad,
        "min_alpha_noise_brightness": b_alpha,
        "min_mad_noise_brightness": b_mad,
        "best_strategy": choice,
        "best_mad": np.nanmin([a_mad, b_mad]) if (not np.isnan(a_mad) or not np.isnan(b_mad)) else np.nan,
    })

df = pd.DataFrame(rows)
df_min = pd.DataFrame(min_rows)

(df.groupby("stage")["p_fake"].agg(["mean","std","count"]).assign(flip_rate=df.groupby("stage")["flip"].mean())).to_csv(
    OUT_DIR / "strategy_stage_summary.csv"
)
df.to_csv(OUT_DIR / "strategy_iteration_results.csv", index=False)
df_min.to_csv(OUT_DIR / "min_change_flip.csv", index=False)

print("Saved:", OUT_DIR / "strategy_iteration_results.csv")
print("Saved:", OUT_DIR / "min_change_flip.csv")

Saved: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\Phase2\results_opt\strategy_iteration_results.csv
Saved: C:\Users\Vedha\feb hack\Feb-2026-Hackathon\Phase2\results_opt\min_change_flip.csv


In [9]:
stage_summary = pd.read_csv(OUT_DIR / "strategy_stage_summary.csv")
results = pd.read_csv(OUT_DIR / "strategy_iteration_results.csv")
min_change = pd.read_csv(OUT_DIR / "min_change_flip.csv")

print("Stage summary:")
display(stage_summary)

flip_blur_freq = results.loc[results.stage == "A_blur_freq", "flip"].mean()
flip_noise_bright = results.loc[results.stage == "B_noise_brightness", "flip"].mean()
mean_hf_drop_blur_freq = results.loc[results.stage == "A_blur_freq", "hf_drop"].mean()
mean_hf_drop_noise_bright = results.loc[results.stage == "B_noise_brightness", "hf_drop"].mean()
mean_contrast_drop_noise = results.loc[results.stage == "B_noise_brightness", "contrast_drop"].mean()

print("\nMinimum-change flip summary:")
print(min_change[["best_strategy", "best_mad"]].groupby("best_strategy").agg(["count", "mean"]))

print("\nAnalysis:")
if mean_hf_drop_blur_freq > mean_hf_drop_noise_bright and flip_blur_freq >= flip_noise_bright:
    print("- Pattern likely suppressed: high-frequency texture/checkerboard-like cues.")
    print("- Model appears to rely heavily on texture-frequency artifacts.")
else:
    print("- Pattern suppression is mixed; texture cues are not the only driver.")

if flip_blur_freq > 0.2:
    print("- Model is sensitive to high-frequency attenuation/noise masking.")
else:
    print("- Sensitivity to high-frequency changes is limited on this sample set.")

if mean_contrast_drop_noise > 0 and flip_noise_bright > 0.1:
    print("- Contrast/brightness artifacts also influence predictions.")
else:
    print("- Contrast shifts alone are weaker than texture-frequency suppression.")

Stage summary:


,stage,mean,std,count,flip_rate
0,A_blur,0.488700,0.114534,48,0.0625
1,A_blur_freq,0.489169,0.114659,48,0.0625
2,B_noise,0.497040,0.117468,48,0.0625
3,B_noise_brightness,0.492937,0.120238,48,0.0625
4,original,0.497360,0.118009,48,0.0000



Minimum-change flip summary:
                 best_mad          
                    count      mean
best_strategy                      
blur_freq               5  0.002386
noise_brightness        4  0.015376
none                    0       NaN

Analysis:
- Pattern likely suppressed: high-frequency texture/checkerboard-like cues.
- Model appears to rely heavily on texture-frequency artifacts.
- Sensitivity to high-frequency changes is limited on this sample set.
- Contrast shifts alone are weaker than texture-frequency suppression.
